# TESS Transit Constraint Validation

This notebook is for **live manual validation** of the raw-TESS transit-fit path in `dataio.tess_photometry`, with preset workflows for:
- KELT-20b
- KELT-9b
- MASCARA-1b
- TOI-1431b
- TOI-1518b
- WASP-12b
- WASP-18b
- WASP-33b
- WASP-76b
- WASP-189b

By default, the notebook is configured to run **all** preset workflows in one execution and write exported `.tbl` files directly to `input/phot/transmission/<planet>/`, not `notebooks/.cache/`.

What is already covered by automated tests:
- retrieval-ready constraint export
- `.tbl` export and parse round-trip
- CLI wiring from `--fit-tess-transit`
- propagation of `quality_bitmask` and `flux_column`

What is **not** covered by automated tests in this environment:
- live `lightkurve` download from MAST
- a real `mlexo` transit fit using your local environment and target choice

Use this notebook to validate the full live path interactively for all supported presets, or narrow `WORKFLOW_SELECTION` to one subset if needed.


In [ ]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault("JAX_PLATFORMS", "cpu")

_candidate_roots = [Path.cwd(), Path.cwd().parent]
for _root in _candidate_roots:
    if (_root / "pipeline").is_dir() and (_root / "config").is_dir():
        REPO_ROOT = _root.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise FileNotFoundError("Could not locate repo root containing pipeline/ and config/")

print(f"Repo root: {REPO_ROOT}")

import matplotlib.pyplot as plt
import numpy as np
import lightkurve as lk
from IPython.display import Markdown, display

import config
from dataio.tess_photometry import (
    TessTransitFitConfig,
    build_joint_tess_dataset,
    download_tess_lightcurves,
    fit_tess_transit_to_bandpass_constraint,
    serialize_tess_fit_summary,
)
from pipeline.retrieval import make_bandpass_constraints_from_tbl

plt.style.use("default")


In [ ]:
# Choose which preset workflows to run.
#   "all" -> run every preset in one notebook execution
#   "kelt20b" -> run a single preset
#   ["kelt20b", "wasp189b"] -> run a subset

PLANET_WORKFLOWS = {
    "kelt20b": {
        "planet": "KELT-20b",
        "ephemeris": "Duck24",
        "tess_target": "KELT-20",
        "period_d": 3.4741039,
        "t0_btjd": 1698.210775,
        "transit_duration_d": 3.54 / 24.0,
        "radius_ratio_guess": 0.116,
        "impact_guess": 0.60,
    },
    "kelt9b": {
        "planet": "KELT-9b",
        "ephemeris": "Gaudi17",
        "tess_target": "KELT-9",
        "period_d": 1.481124,
        "t0_btjd": 2000.410541,
        "transit_duration_d": 0.163158,
        "radius_ratio_guess": 0.0804,
        "impact_guess": 0.18,
    },
    "mascara1b": {
        "planet": "MASCARA-1b",
        "ephemeris": "Talens17",
        "tess_target": "MASCARA-1",
        "period_d": 2.148774,
        "t0_btjd": 1998.943734,
        "transit_duration_d": 0.176083,
        "radius_ratio_guess": 0.0771,
        "impact_guess": 0.11,
    },
    "toi1431b": {
        "planet": "TOI-1431b",
        "ephemeris": "Addison21",
        "tess_target": "TOI-1431",
        "period_d": 2.650237,
        "t0_btjd": 1998.900596,
        "transit_duration_d": 0.103708,
        "radius_ratio_guess": 0.0780,
        "impact_guess": 0.88,
    },
    "toi1518b": {
        "planet": "TOI-1518b",
        "ephemeris": "Cabot21",
        "tess_target": "TOI-1518",
        "period_d": 1.902603,
        "t0_btjd": 2000.140791,
        "transit_duration_d": 0.098542,
        "radius_ratio_guess": 0.0966,
        "impact_guess": 0.90,
    },
    "wasp12b": {
        "planet": "WASP-12b",
        "ephemeris": "Ivshina22",
        "tess_target": "WASP-12",
        "period_d": 1.091419,
        "t0_btjd": 2000.169207,
        "transit_duration_d": 0.126700,
        "radius_ratio_guess": 0.1202,
        "impact_guess": 0.36,
    },
    "wasp18b": {
        "planet": "WASP-18b",
        "ephemeris": "Cortes-Zuleta20",
        "tess_target": "WASP-18",
        "period_d": 0.941452,
        "t0_btjd": 2000.290952,
        "transit_duration_d": 0.092083,
        "radius_ratio_guess": 0.1013,
        "impact_guess": 0.41,
    },
    "wasp33b": {
        "planet": "WASP-33b",
        "ephemeris": "Ivshina22",
        "tess_target": "WASP-33",
        "period_d": 1.219870,
        "t0_btjd": 2000.008220,
        "transit_duration_d": 0.118917,
        "radius_ratio_guess": 0.1060,
        "impact_guess": 0.21,
    },
    "wasp76b": {
        "planet": "WASP-76b",
        "ephemeris": "West16",
        "tess_target": "WASP-76",
        "period_d": 1.809886,
        "t0_btjd": 2000.052898,
        "transit_duration_d": 0.153917,
        "radius_ratio_guess": 0.1063,
        "impact_guess": 0.14,
    },
    "wasp189b": {
        "planet": "WASP-189b",
        "ephemeris": "Anderson18",
        "tess_target": "WASP-189",
        "period_d": 2.724031,
        "t0_btjd": 2000.090528,
        "transit_duration_d": 0.180567,
        "radius_ratio_guess": 0.0689,
        "impact_guess": 0.48,
    },
}

WORKFLOW_SELECTION = "all"

QUALITY_BITMASK = "default"   # one of none/default/hard/hardest or an integer bitmask
FLUX_COLUMN = "pdcsap_flux"   # try "sap_flux" if you want to compare products
OBSERVABLE = "radius_ratio"   # or "transit_depth"
CONSTRAINT_NAME = "tess_transit"
PHOTON_WEIGHTED = False
WRITE_TBL = True


def _choose_finite(value, fallback):
    try:
        candidate = float(value)
    except (TypeError, ValueError):
        return float(fallback), "preset"
    if np.isfinite(candidate):
        return candidate, "config"
    return float(fallback), "preset"


def _resolve_workflow_keys(selection):
    if selection == "all":
        return list(PLANET_WORKFLOWS.keys())
    if isinstance(selection, str):
        return [selection]
    return list(selection)


def _build_runtime(workflow_key):
    workflow = PLANET_WORKFLOWS[workflow_key]
    planet = workflow["planet"]
    ephemeris = workflow["ephemeris"]
    tess_target = workflow["tess_target"]
    sectors = workflow.get("sectors")
    mission = workflow.get("mission", "TESS")
    author = workflow.get("author", "SPOC")
    exptime_s = workflow.get("exptime_s", 120)

    planet_slug = planet.lower().replace("-", "").replace(" ", "")
    phot_output_dir = REPO_ROOT / "input" / "phot" / "transmission" / planet_slug
    phot_output_dir.mkdir(parents=True, exist_ok=True)
    tbl_output_path = phot_output_dir / f"{planet_slug}_tess_bandpass.tbl"

    params = config.get_params(planet=planet, ephemeris=ephemeris)
    period_d, period_source = _choose_finite(params.get("period"), workflow["period_d"])
    t0_bjd, t0_source = _choose_finite(params.get("epoch"), workflow["t0_btjd"] + 2457000.0)
    t0_btjd = t0_bjd - 2457000.0
    transit_duration_d, duration_source = _choose_finite(params.get("duration"), workflow["transit_duration_d"])
    radius_ratio_guess, radius_ratio_source = _choose_finite(params.get("rp_rs"), workflow["radius_ratio_guess"])
    impact_guess, impact_source = _choose_finite(params.get("b"), workflow["impact_guess"])

    fit_config = TessTransitFitConfig(
        target=tess_target,
        period_d=period_d,
        t0_btjd=t0_btjd,
        transit_duration_d=transit_duration_d,
        radius_ratio_guess=radius_ratio_guess,
        impact_guess=impact_guess,
        mission=mission,
        author=author,
        exptime_s=exptime_s,
        quality_bitmask=QUALITY_BITMASK,
        flux_column=FLUX_COLUMN,
        sectors=None if sectors is None else tuple(sectors),
        planet_name=planet,
        reference=ephemeris,
        note="Notebook-generated TESS bandpass constraint",
    )

    return {
        "workflow_key": workflow_key,
        "planet": planet,
        "planet_slug": planet_slug,
        "ephemeris": ephemeris,
        "tess_target": tess_target,
        "sectors": sectors,
        "period_d": period_d,
        "period_source": period_source,
        "t0_bjd": t0_bjd,
        "t0_source": t0_source,
        "t0_btjd": t0_btjd,
        "transit_duration_d": transit_duration_d,
        "duration_source": duration_source,
        "radius_ratio_guess": radius_ratio_guess,
        "radius_ratio_source": radius_ratio_source,
        "impact_guess": impact_guess,
        "impact_source": impact_source,
        "tbl_output_path": tbl_output_path,
        "fit_config": fit_config,
    }


SELECTED_WORKFLOW_KEYS = _resolve_workflow_keys(WORKFLOW_SELECTION)
WORKFLOW_RUNS = {key: _build_runtime(key) for key in SELECTED_WORKFLOW_KEYS}

workflow_summary = {
    key: {
        "planet": value["planet"],
        "ephemeris": value["ephemeris"],
        "tess_target": value["tess_target"],
    }
    for key, value in PLANET_WORKFLOWS.items()
}

selected_summary = {
    key: {
        "planet": runtime["planet"],
        "ephemeris": runtime["ephemeris"],
        "tess_target": runtime["tess_target"],
        "sectors": runtime["sectors"],
        "period_d": runtime["period_d"],
        "period_source": runtime["period_source"],
        "t0_bjd": runtime["t0_bjd"],
        "t0_source": runtime["t0_source"],
        "transit_duration_d": runtime["transit_duration_d"],
        "duration_source": runtime["duration_source"],
        "radius_ratio_guess": runtime["radius_ratio_guess"],
        "radius_ratio_source": runtime["radius_ratio_source"],
        "impact_guess": runtime["impact_guess"],
        "impact_source": runtime["impact_source"],
        "tbl_output_path": str(runtime["tbl_output_path"]),
    }
    for key, runtime in WORKFLOW_RUNS.items()
}

display(Markdown("## Available preset workflows"))
print(json.dumps(workflow_summary, indent=2))

display(Markdown("## Selected workflows for this execution"))
print(json.dumps(
    {
        "workflow_selection": WORKFLOW_SELECTION,
        "selected_workflow_keys": SELECTED_WORKFLOW_KEYS,
        "observable": OBSERVABLE,
        "flux_column": FLUX_COLUMN,
        "quality_bitmask": QUALITY_BITMASK,
        "write_tbl": WRITE_TBL,
        "selected_workflows": selected_summary,
    },
    indent=2,
))


In [ ]:
SEARCH_SUMMARY = {}

for workflow_key in SELECTED_WORKFLOW_KEYS:
    runtime = WORKFLOW_RUNS[workflow_key]
    fit_config = runtime["fit_config"]
    search_kwargs = {
        "mission": fit_config.mission,
        "author": fit_config.author,
        "exptime": fit_config.exptime_s,
    }
    if fit_config.sectors is not None:
        search_kwargs["sector"] = list(fit_config.sectors)

    search = lk.search_lightcurve(fit_config.target, **search_kwargs)
    runtime["search"] = search
    SEARCH_SUMMARY[workflow_key] = {
        "planet": runtime["planet"],
        "tess_target": fit_config.target,
        "matches": int(len(search)),
        "search_kwargs": search_kwargs,
    }

display(Markdown("## Lightkurve search summary"))
print(json.dumps(SEARCH_SUMMARY, indent=2))

if len(SELECTED_WORKFLOW_KEYS) == 1:
    only_runtime = WORKFLOW_RUNS[SELECTED_WORKFLOW_KEYS[0]]
    display(Markdown(f"## Detailed search results for {only_runtime['planet']}"))
    display(only_runtime["search"])


In [ ]:
DATASET_SUMMARY = {}
n_workflows = len(SELECTED_WORKFLOW_KEYS)
ncols = 1 if n_workflows == 1 else 2
nrows = int(np.ceil(n_workflows / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12 * ncols / 2, 4.5 * nrows), squeeze=False)

for axis_index, workflow_key in enumerate(SELECTED_WORKFLOW_KEYS):
    ax = axes.flat[axis_index]
    runtime = WORKFLOW_RUNS[workflow_key]
    fit_config = runtime["fit_config"]

    lc_collection = download_tess_lightcurves(fit_config)
    sector_labels = [lc.meta.get("MISSION", f"sector_{idx + 1}") for idx, lc in enumerate(lc_collection)]
    dataset = build_joint_tess_dataset(
        lc_collection,
        period_d=fit_config.period_d,
        t0_btjd=fit_config.t0_btjd,
        transit_duration_d=fit_config.transit_duration_d,
        model_window_d=fit_config.model_window_d,
        flatten_window_length=fit_config.flatten_window_length,
        outlier_sigma=fit_config.outlier_sigma,
    )
    phase_days = ((dataset["time"] - fit_config.t0_btjd + 0.5 * fit_config.period_d) % fit_config.period_d) - 0.5 * fit_config.period_d

    runtime["lc_collection"] = lc_collection
    runtime["dataset"] = dataset
    runtime["phase_days"] = phase_days
    runtime["sector_labels"] = sector_labels

    DATASET_SUMMARY[workflow_key] = {
        "planet": runtime["planet"],
        "n_sectors": int(dataset["n_sectors"]),
        "sector_labels": list(dataset["sector_labels"]),
        "sector_counts": [int(x) for x in dataset["sector_counts"]],
        "n_cadences_total": int(dataset["time"].size),
    }

    for sector_id, label in enumerate(dataset["sector_labels"]):
        rows = dataset["sector_rows"][sector_id]
        ax.scatter(phase_days[rows], dataset["flux"][rows], s=8, alpha=0.45, label=label)
    ax.axvline(0.0, color="black", lw=1.0, alpha=0.35)
    ax.set_title(runtime["planet"])
    ax.set_xlabel("Phase [days]")
    ax.set_ylabel("Normalized flux - 1")
    if len(dataset["sector_labels"]) <= 4:
        ax.legend(loc="best", fontsize=7)

for axis_index in range(n_workflows, nrows * ncols):
    axes.flat[axis_index].axis("off")

display(Markdown("## Downloaded light curves and pre-fit dataset summary"))
print(json.dumps(DATASET_SUMMARY, indent=2))

plt.tight_layout()
plt.show()


In [ ]:
FIT_SUMMARY = {}

for workflow_key in SELECTED_WORKFLOW_KEYS:
    runtime = WORKFLOW_RUNS[workflow_key]
    result = fit_tess_transit_to_bandpass_constraint(
        runtime["fit_config"],
        lc_collection=runtime["lc_collection"],
        observable=OBSERVABLE,
        constraint_name=CONSTRAINT_NAME,
        photon_weighted=True if PHOTON_WEIGHTED else None,
        tbl_path=runtime["tbl_output_path"] if WRITE_TBL else None,
    )

    summary = serialize_tess_fit_summary(result)
    runtime["result"] = result
    runtime["summary"] = summary

    FIT_SUMMARY[workflow_key] = {
        "planet": runtime["planet"],
        "constraint": summary["constraint"],
        "transit_depth_percent": summary["fit"]["summary_stats"]["transit_depth_percent"],
        "r": summary["fit"]["summary_stats"]["r"],
        "b": summary["fit"]["summary_stats"]["b"],
        "duration": summary["fit"]["summary_stats"]["duration"],
        "tbl_output_path": str(runtime["tbl_output_path"]),
    }

display(Markdown("## Retrieval-ready constraints and selected posterior summaries"))
print(json.dumps(FIT_SUMMARY, indent=2))

if WRITE_TBL:
    display(Markdown("## Written TESS bandpass tables"))
    print(json.dumps({key: str(runtime["tbl_output_path"]) for key, runtime in WORKFLOW_RUNS.items()}, indent=2))


In [ ]:
for workflow_key in SELECTED_WORKFLOW_KEYS:
    runtime = WORKFLOW_RUNS[workflow_key]
    best_fit = runtime["result"].best_fit
    summary = runtime["summary"]

    display(Markdown(f"## {runtime['planet']} best-fit phase curve"))

    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    axes[0].errorbar(
        best_fit["phase_days"],
        best_fit["detrended_flux"],
        yerr=runtime["result"].dataset["flux_err"],
        fmt=".",
        alpha=0.25,
        ms=4,
        color="tab:blue",
    )
    phase_order = np.argsort(best_fit["phase_grid"])
    axes[0].plot(
        best_fit["phase_grid"][phase_order],
        best_fit["phase_model_grid"][phase_order],
        color="black",
        lw=2,
        label="Transit model",
    )
    axes[0].set_ylabel("Detrended flux")
    axes[0].set_title(runtime["planet"])
    axes[0].legend(loc="best")

    axes[1].scatter(best_fit["phase_days"], best_fit["residual_flux"], s=10, alpha=0.4, color="tab:orange")
    axes[1].axhline(0.0, color="black", lw=1.0, alpha=0.4)
    axes[1].set_xlabel("Phase [days]")
    axes[1].set_ylabel("Residual flux")

    plt.tight_layout()
    plt.show()

    print(json.dumps(
        {
            "planet": runtime["planet"],
            "emcee_acceptance_fraction": summary["fit"]["emcee_acceptance_fraction"],
            "emcee_parallel": summary["fit"]["emcee_parallel"],
            "emcee_worker_count": summary["fit"]["emcee_worker_count"],
        },
        indent=2,
    ))


In [ ]:
if WRITE_TBL:
    ROUNDTRIP_SUMMARY = {}
    for workflow_key in SELECTED_WORKFLOW_KEYS:
        runtime = WORKFLOW_RUNS[workflow_key]
        parsed_constraints = make_bandpass_constraints_from_tbl(runtime["tbl_output_path"])
        runtime["parsed_constraints"] = parsed_constraints
        ROUNDTRIP_SUMMARY[workflow_key] = parsed_constraints

    display(Markdown("## Constraint parsed back through retrieval helpers"))
    print(json.dumps(ROUNDTRIP_SUMMARY, indent=2))
else:
    print("WRITE_TBL is False; no .tbl round-trip performed.")


## Next step into retrieval

If the live fits above look good, each exported file is already in the canonical input tree:

- `input/phot/transmission/<planet>/<planet>_tess_bandpass.tbl`

You can then point the CLI at any one of them with `--bandpass-tbl transmission/<planet>/<planet>_tess_bandpass.tbl`, or use the in-memory handoff directly from `runtime["result"].bandpass_constraint` for the corresponding workflow.
